# TASK03 — SAM 3D Body → MHR → clad-body: Colab GPU end-to-end smoke test

**Technical smoke test only. Not an anthropometric accuracy benchmark — do not treat any
measurement produced here as validated.** See `docs/experiments/TASK03_COLAB_END_TO_END_SMOKE_TEST.md`.

Requirements before running:
1. Colab runtime with a GPU: `Runtime > Change runtime type > T4 GPU` (or better, free tier).
2. A Colab Secret named exactly `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face
   **read** token that has been granted approved access to `facebook/sam-3d-body-dinov3` and
   `facebook/sam-3d-body-vith`. Enable notebook access for the secret.
3. Run cells top-to-bottom. A cell raises with a clear message and the notebook stops if a
   precondition (GPU, token, checkpoint access) is not met — nothing downstream is faked.

## 1. Environment inspection

In [ ]:
import platform, shutil, subprocess, sys, time

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print('Python:', platform.python_version())
print('Platform:', platform.platform())

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 1e9
except ImportError:
    ram_gb = None
print('System RAM (GB):', round(ram_gb, 1) if ram_gb else 'unknown (psutil not available)')

disk = shutil.disk_usage('/')
print(f'Disk: {disk.free/1e9:.1f} GB free / {disk.total/1e9:.1f} GB total')

gpu_query = sh('nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader 2>/dev/null')
gpu_available = bool(gpu_query)
print('nvidia-smi GPU query:', gpu_query if gpu_available else 'NONE DETECTED')

cuda_version = None
torch_version = None
try:
    import torch
    torch_version = torch.__version__
    cuda_version = torch.version.cuda
    print('PyTorch:', torch_version, '| CUDA build:', cuda_version, '| torch.cuda.is_available():', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('CUDA device:', torch.cuda.get_device_name(0))
        print('VRAM total (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
        gpu_available = gpu_available and True
    else:
        gpu_available = False
except ImportError:
    print('torch not importable yet in this kernel')
    gpu_available = False

if not gpu_available:
    raise RuntimeError(
        'NO_GPU: no usable NVIDIA GPU detected in this Colab runtime. '
        'Go to Runtime > Change runtime type, select a GPU, then Runtime > Restart and run all. '
        'Per Task 03 section 1, this notebook does not attempt full inference on CPU.'
    )

## 2. Secure HF_TOKEN retrieval

Read only from Colab Secrets. Never printed, logged, written to a file, or passed to
`huggingface_hub.login()` (which persists it to `~/.huggingface/token` on disk) — only passed
explicitly as a `token=` argument to the specific API calls below that need it.

In [ ]:
from google.colab import userdata

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError(
        "HF_AUTH_FAILURE: could not read the 'HF_TOKEN' Colab Secret. Add it via the key icon "
        "in the left sidebar, name it exactly HF_TOKEN, and enable notebook access. "
        f"Original error: {exc!r}"
    )

if not HF_TOKEN:
    raise RuntimeError('HF_AUTH_FAILURE: HF_TOKEN secret is empty.')

print(f'HF_TOKEN retrieved from Colab Secrets: OK ({len(HF_TOKEN)} chars, value not shown)')
hf_token_present = True  # only a boolean is kept at module level from here on, never the token's value in any printed/saved structure

## 3. Repository checkout/setup

In [ ]:
import os

REPO_URL = 'https://github.com/eliyahumines-dot/mtm-body-checker.git'
REPO_BRANCH = 'claude/body-measurement-feasibility-4qgdrz'  # update once merged to main
REPO_DIR = '/content/mtm-body-checker'

if not os.path.isdir(REPO_DIR):
    rc = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
                        capture_output=True, text=True)
    print(rc.stdout[-1000:], rc.stderr[-1000:])
else:
    print('Repo already present at', REPO_DIR)

EXPERIMENT_DIR = f'{REPO_DIR}/experiments/sam3d_mhr_clad_smoke'
if not os.path.isdir(EXPERIMENT_DIR):
    raise RuntimeError(
        f'{EXPERIMENT_DIR} not found. If this repo is private and the clone above failed, '
        f'manually upload the experiments/sam3d_mhr_clad_smoke/ folder to {EXPERIMENT_DIR} '
        f'(Colab file browser, drag-and-drop) and re-run this cell.'
    )
sys.path.insert(0, EXPERIMENT_DIR)
print('Reusing Task 02/03 code from:', EXPERIMENT_DIR)

from decision_gate import PipelineState, FailureCategory, DecisionGate, classify
from adapter import AdapterError, sam3d_output_to_clad_params, warn_if_scale_params_would_be_misused, \
    EXPECTED_SHAPE_PARAMS_LEN, EXPECTED_MHR_MODEL_PARAMS_LEN
from run import measure_via_subprocess

state = PipelineState(gpu_available=gpu_available)

## 4. Dependency installation

Two separate environments, deliberately:
- **Main Colab kernel** keeps Colab's pre-installed CUDA-enabled `torch` untouched, and adds
  only SAM 3D Body's own dependencies (incl. `detectron2`, `MoGe`) — this is what runs GPU inference.
- **A separate CPU-torch venv** hosts `clad-body[mhr]` (`pymomentum-cpu`). Task 02 found
  `pymomentum-cpu` ("linked against CPU PyTorch" per its own PyPI description) segfaults when
  loaded in the same process as a CUDA-tagged torch build — an ABI mismatch, not a clad-body bug.
  Colab has unrestricted internet access to `download.pytorch.org` (Task 02's sandbox did not),
  so a properly CPU-only torch build is installable here. This is the smallest reproducible
  correction: give `pymomentum-cpu` the specific torch build it expects, in its own environment,
  reusing `clad-body`'s own existing subprocess-isolation design (it already shells out for an
  unrelated import-order reason) rather than patching any third-party source.

In [ ]:
def run_shell(cmd, timeout=1800, cwd=None):
    print(f'$ {cmd}')
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, cwd=cwd)
    if proc.returncode != 0:
        print(proc.stdout[-3000:])
        print(proc.stderr[-3000:])
    return proc.returncode == 0

dependencies_installed = True

# 4a. Main kernel: SAM 3D Body's own dependencies (GPU-side, native Colab torch untouched)
SAM3D_DIR = '/content/sam-3d-body'
if not os.path.isdir(SAM3D_DIR):
    dependencies_installed &= run_shell(f'git clone --depth 1 https://github.com/facebookresearch/sam-3d-body.git {SAM3D_DIR}')

sam3d_pip_deps = (
    'pytorch-lightning pyrender opencv-python yacs scikit-image einops timm dill pandas rich '
    'hydra-core hydra-submitit-launcher hydra-colorlog pyrootutils webdataset chumpy '
    'networkx==3.2.1 roma joblib seaborn wandb appdirs jsonlines xtcocotools loguru optree '
    'fvcore pycocotools huggingface_hub'
)
dependencies_installed &= run_shell(f'pip install -q {sam3d_pip_deps}')
dependencies_installed &= run_shell("pip install -q 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps")
dependencies_installed &= run_shell('pip install -q git+https://github.com/microsoft/MoGe.git')

sys.path.insert(0, SAM3D_DIR)
print('SAM 3D Body dependency install:', 'OK' if dependencies_installed else 'FAILED')

In [ ]:
# 4b. Separate CPU-torch venv for clad-body[mhr] / pymomentum-cpu (the Task 02 fix)
CLAD_VENV_DIR = '/content/clad_env'
clad_venv_python = f'{CLAD_VENV_DIR}/bin/python3'

t0 = time.time()
clad_env_ok = True
clad_env_ok &= run_shell(f'python3 -m venv {CLAD_VENV_DIR}')
clad_env_ok &= run_shell(f'{clad_venv_python} -m pip install -q --upgrade pip')
clad_env_ok &= run_shell(f'{clad_venv_python} -m pip install -q torch --index-url https://download.pytorch.org/whl/cpu')
clad_env_ok &= run_shell(f"{clad_venv_python} -m pip install -q 'clad-body[mhr]'")
clad_env_setup_time_s = round(time.time() - t0, 1)
print('clad-body CPU venv setup:', 'OK' if clad_env_ok else 'FAILED', f'({clad_env_setup_time_s}s)')

if not clad_env_ok:
    state.add_failure(FailureCategory.CUDA_PYTORCH_MISMATCH)
    dependencies_installed = False

# MHR body-model/rig assets: PyPI's `mhr` package omits the download_assets console script
# (Task 02 finding) -- fetch the same public, unauthenticated GitHub release archive directly.
if clad_env_ok:
    import glob
    site_packages_matches = glob.glob(f'{CLAD_VENV_DIR}/lib/python*/site-packages')
    assets_dir = f'{site_packages_matches[0]}/assets' if site_packages_matches else None
    if assets_dir and not os.path.isdir(assets_dir):
        run_shell(f'mkdir -p {assets_dir}')
        clad_env_ok &= run_shell(
            'curl -sSL -o /content/mhr_assets.zip '
            'https://github.com/facebookresearch/MHR/releases/latest/download/assets.zip'
        )
        clad_env_ok &= run_shell(f'unzip -q -o /content/mhr_assets.zip -d {assets_dir}')
        nested = f'{assets_dir}/assets'
        if os.path.isdir(nested):
            run_shell(f'mv {nested}/* {assets_dir}/ && rmdir {nested}')
    print('MHR assets ready at', assets_dir)

dependencies_installed = dependencies_installed and clad_env_ok
state.dependencies_installed = dependencies_installed
print('Overall dependency install status:', 'OK' if dependencies_installed else 'FAILED')

## 5. Checkpoint authentication test

Verify authenticated access to both official repos *before* downloading anything. Per
upstream `INSTALL.md`, `facebook/sam-3d-body-dinov3` is the checkpoint used in the README's
own example command — treated here as the documented practical default; only it is downloaded
unless the section 15 comparison is explicitly enabled.

In [ ]:
from huggingface_hub import HfApi

CHECKPOINT_REPOS = ['facebook/sam-3d-body-dinov3', 'facebook/sam-3d-body-vith']
PRIMARY_CHECKPOINT_REPO = 'facebook/sam-3d-body-dinov3'  # upstream README's own default example

api = HfApi()
access_status = {}
for repo_id in CHECKPOINT_REPOS:
    try:
        api.model_info(repo_id, token=HF_TOKEN)
        access_status[repo_id] = 'accessible'
    except Exception as exc:
        access_status[repo_id] = f'BLOCKED: {type(exc).__name__}'  # exception text can echo request headers; keep only the type

print(access_status)

hf_auth_ok = access_status.get(PRIMARY_CHECKPOINT_REPO) == 'accessible'
state.hf_auth_ok = hf_auth_ok
if not hf_auth_ok:
    state.add_failure(FailureCategory.HF_AUTH_FAILURE)
    raise RuntimeError(
        f'CHECKPOINT_ACCESS_BLOCKED: authenticated access to {PRIMARY_CHECKPOINT_REPO} not confirmed '
        f'({access_status.get(PRIMARY_CHECKPOINT_REPO)}). Confirm HF_TOKEN is valid and that access '
        f'has actually been approved for this exact repo on huggingface.co.'
    )

## 6-11. Input image, SAM 3D Body inference, schema inspection, adapter, MHR reconstruction, measurement

Factored into one function so the (optional) second-checkpoint comparison in section 15 reuses it
exactly rather than duplicating ~11 sections of pipeline code.

In [ ]:
import json, tempfile, traceback

def download_checkpoint(repo_id):
    from huggingface_hub import snapshot_download
    local_dir = f"/content/checkpoints/{repo_id.split('/')[-1]}"
    t0 = time.time()
    snapshot_download(repo_id=repo_id, local_dir=local_dir, token=HF_TOKEN)
    download_time_s = round(time.time() - t0, 1)
    size_bytes = int(sh(f'du -sb {local_dir}').split()[0]) if os.path.isdir(local_dir) else None
    return local_dir, download_time_s, size_bytes


def run_full_pipeline_for_checkpoint(repo_id, image_path, known_height_cm=None, record_to_state=False):
    """Run download -> SAM 3D Body inference -> schema check -> adapter -> MHR -> measure
    for one checkpoint repo. Returns a result dict; never raises -- failures are recorded in it.
    If record_to_state, also updates the module-level `state` PipelineState (used for the
    PRIMARY checkpoint only -- the section 15 comparison checkpoint does not affect the gate).
    """
    r = {
        'checkpoint_repo': repo_id, 'checkpoint_downloaded': False, 'checkpoint_download_time_s': None,
        'checkpoint_size_bytes': None, 'sam3d_load_time_s': None, 'sam3d_inference_time_s': None,
        'peak_vram_mb': None, 'sam3d_inference_ok': False, 'person_detected': False,
        'output_schema': {}, 'mhr_schema_valid': False, 'measurement_extraction_time_s': None,
        'raw_result': None, 'calibrated_result': None, 'warnings': [], 'failures': [],
    }
    try:
        ckpt_dir, dl_time, size_bytes = download_checkpoint(repo_id)
        r['checkpoint_downloaded'] = True
        r['checkpoint_download_time_s'] = dl_time
        r['checkpoint_size_bytes'] = size_bytes
    except Exception as exc:
        r['failures'].append(f'CHECKPOINT_ACCESS_FAILURE: {exc!r}')
        if record_to_state:
            state.checkpoint_downloaded = False
            state.add_failure(FailureCategory.CHECKPOINT_ACCESS_FAILURE)
        return r
    if record_to_state:
        state.checkpoint_downloaded = True

    person_output = None
    try:
        from sam_3d_body import load_sam_3d_body, SAM3DBodyEstimator
        device = torch.device('cuda')
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        ckpt_path = f'{ckpt_dir}/model.ckpt'
        mhr_asset_path = f'{ckpt_dir}/assets/mhr_model.pt'
        model, model_cfg = load_sam_3d_body(ckpt_path, device=device, mhr_path=mhr_asset_path)
        r['sam3d_load_time_s'] = round(time.time() - t0, 1)
        estimator = SAM3DBodyEstimator(sam_3d_body_model=model, model_cfg=model_cfg)
        t1 = time.time()
        outputs = estimator.process_one_image(image_path)
        r['sam3d_inference_time_s'] = round(time.time() - t1, 2)
        r['peak_vram_mb'] = round(torch.cuda.max_memory_allocated() / 1e6, 1)
        if outputs:
            person_output = outputs[0]
            r['sam3d_inference_ok'] = True
            r['person_detected'] = True
        else:
            r['sam3d_inference_ok'] = True  # ran without error, just found nobody
            r['warnings'].append('SAM 3D Body ran but detected no person in the image')
    except Exception as exc:
        r['failures'].append(f'SAM3D_INFERENCE_FAILURE: {exc!r}')
        traceback.print_exc()
        if record_to_state:
            state.sam3d_inference_ok = False
            state.add_failure(FailureCategory.SAM3D_INFERENCE_FAILURE)
        return r
    if record_to_state:
        state.sam3d_inference_ok = r['sam3d_inference_ok']

    if person_output is None:
        return r  # ran, but nobody detected -- nothing further to extract

    for k, v in person_output.items():
        try:
            shape = tuple(v.shape) if hasattr(v, 'shape') else (len(v) if hasattr(v, '__len__') else None)
        except Exception:
            shape = '?'
        dtype = str(v.dtype) if hasattr(v, 'dtype') else type(v).__name__
        r['output_schema'][k] = {'shape': list(shape) if isinstance(shape, tuple) else shape, 'dtype': dtype}

    warn = warn_if_scale_params_would_be_misused(person_output)
    if warn:
        r['warnings'].append(warn)

    clad_params = None
    try:
        clad_params = sam3d_output_to_clad_params(person_output)
        r['mhr_schema_valid'] = True
        assert len(clad_params['shape_params']) == EXPECTED_SHAPE_PARAMS_LEN
        assert len(clad_params['mhr_model_params']) == EXPECTED_MHR_MODEL_PARAMS_LEN
    except AdapterError as exc:
        r['failures'].append(f'MHR_SCHEMA_FAILURE: {exc}')
        if record_to_state:
            state.mhr_schema_valid = False
            state.add_failure(FailureCategory.MHR_SCHEMA_FAILURE)
        return r
    if record_to_state:
        state.mhr_schema_valid = True

    with tempfile.NamedTemporaryFile(mode='w', suffix='_sam3d_mhr_params.json', delete=False) as f:
        json.dump(clad_params, f)
        params_json_path = f.name

    from _mhr_measure_worker import parse_last_stage

    def _measure(known_height):
        w, fl = [], []
        t0 = time.time()
        status, res = measure_via_subprocess(
            params_json_path, known_height_cm=known_height, warnings=w, failures=fl,
            python_executable=clad_venv_python,
        )
        elapsed = round(time.time() - t0, 2)
        return status, res, w, fl, elapsed

    status, raw_res, w, fl, elapsed = _measure(None)
    r['measurement_extraction_time_s'] = elapsed
    r['warnings'].extend(w)
    r['failures'].extend(fl)

    if status == 'ok':
        r['raw_result'] = raw_res
        if record_to_state:
            state.mhr_reconstruction_ok = True
            state.clad_body_measure_ok = True
            state.measurements = raw_res['measurements_cm']
    else:
        stage = parse_last_stage(' '.join(fl))
        category = FailureCategory.PYMOMENTUM_FAILURE if stage == 'mhr_reconstruction' else FailureCategory.CLAD_BODY_FAILURE
        r['failures'].append(f'{category.value}: worker status={status}, last stage={stage}')
        if record_to_state:
            if stage == 'mhr_reconstruction':
                state.mhr_reconstruction_ok = False
            else:
                state.mhr_reconstruction_ok = True
                state.clad_body_measure_ok = False
            state.add_failure(category)
        return r

    if known_height_cm is not None:
        status2, cal_res, w2, fl2, elapsed2 = _measure(known_height_cm)
        r['warnings'].extend(w2)
        r['failures'].extend(fl2)
        if status2 == 'ok':
            r['calibrated_result'] = cal_res
        else:
            r['warnings'].append(f'known-height calibration run failed: {status2}')

    try:
        os.unlink(params_json_path)
    except OSError:
        pass
    return r

print('run_full_pipeline_for_checkpoint() defined.')

## 6. Input image selection

Defaults to a public sample image bundled in the official `sam-3d-body` repo (no personal data
involved, safe to run without uploading anything). Set `USE_SAMPLE_IMAGE = False` and re-run to
upload your own photo instead — an uploaded photo stays only in this Colab runtime's `/content`
and is never written into the cloned git repo.

In [ ]:
USE_SAMPLE_IMAGE = True

SAMPLE_IMAGE = f'{SAM3D_DIR}/assets/qualitative_comparisons/sample1/input_bbox.png'

if USE_SAMPLE_IMAGE:
    input_image_path = SAMPLE_IMAGE
    used_sample_image = True
    if not os.path.exists(input_image_path):
        raise RuntimeError(f'Bundled sample image not found at {input_image_path} -- check the SAM3D_DIR clone in section 4.')
else:
    from google.colab import files
    uploaded = files.upload()
    input_image_path = f'/content/{next(iter(uploaded))}'
    used_sample_image = False

print('Input image:', input_image_path, '(public sample)' if used_sample_image else '(user-uploaded, not committed)')

## 7-11. Run the primary checkpoint end-to-end

`KNOWN_HEIGHT_CM` is optional (section 12 of the task) — set it before running this cell to also
get a height-calibrated result alongside the raw one; leave `None` to skip calibration.

In [ ]:
KNOWN_HEIGHT_CM = None  # e.g. 178.0 -- customer-reported height in cm, or None to skip calibration

primary = run_full_pipeline_for_checkpoint(
    PRIMARY_CHECKPOINT_REPO, input_image_path, known_height_cm=KNOWN_HEIGHT_CM, record_to_state=True,
)

print('checkpoint:', primary['checkpoint_repo'])
print('sam3d_inference_ok:', primary['sam3d_inference_ok'], '| person_detected:', primary['person_detected'])
print('mhr_schema_valid:', primary['mhr_schema_valid'])
print('raw_result present:', primary['raw_result'] is not None)
print('calibrated_result present:', primary['calibrated_result'] is not None)
for w in primary['warnings']:
    print('WARNING:', w)
for f_ in primary['failures']:
    print('FAILURE:', f_)

### 8. Actual SAM 3D Body output schema (as observed, not assumed from docs)

In [ ]:
for k, v in primary['output_schema'].items():
    print(f"  {k}: shape={v['shape']} dtype={v['dtype']}")
if not primary['output_schema']:
    print('No output schema captured (inference did not produce a person_output -- see failures above).')

### 9. shape_params / mhr_model_params vs. raw scale_params (Task 02's trap, confirmed against real output)

In [ ]:
sp = primary['output_schema'].get('shape_params')
mp = primary['output_schema'].get('mhr_model_params')
scp = primary['output_schema'].get('scale_params')
print('shape_params shape:', sp['shape'] if sp else 'n/a', f'(expected [{EXPECTED_SHAPE_PARAMS_LEN}])')
print('mhr_model_params shape:', mp['shape'] if mp else 'n/a', f'(expected [{EXPECTED_MHR_MODEL_PARAMS_LEN}])')
if scp:
    print('scale_params shape (NOT passed to clad-body -- see adapter.py docstring):', scp['shape'])

### 10-11. MHR reconstruction + clad-body measurements (raw and, if requested, height-calibrated)

In [ ]:
raw_measurements_cm = primary['raw_result']['measurements_cm'] if primary['raw_result'] else None
raw_body_height_cm = primary['raw_result']['raw_body_height_cm'] if primary['raw_result'] else None
calibrated_measurements_cm = primary['calibrated_result']['measurements_cm'] if primary['calibrated_result'] else None
calibrated_scale_factor = primary['calibrated_result']['rescale_factor'] if primary['calibrated_result'] else None

print('RAW body height (cm):', raw_body_height_cm)
print('RAW measurements (cm):', json.dumps(raw_measurements_cm, indent=2) if raw_measurements_cm else None)
if calibrated_measurements_cm:
    print(f'scale_factor = known_height / predicted_height = {KNOWN_HEIGHT_CM} / {raw_body_height_cm:.2f} = {calibrated_scale_factor:.4f}')
    print('CALIBRATED measurements (cm):', json.dumps(calibrated_measurements_cm, indent=2))

### MTM measurement terminology mapping (reused from Task 02, unchanged)

In [ ]:
from mtm_mapping import MTM_MEASUREMENT_MAP

if raw_measurements_cm:
    for m in MTM_MEASUREMENT_MAP:
        val = raw_measurements_cm.get(m.clad_body_key) if m.clad_body_key else None
        print(f'{m.mtm_name:32s} <- {str(m.clad_body_key):20s} = {val}  [{m.confidence}]')
else:
    print('No raw measurements available to map.')

## 12. Known-height calibration — note

Already applied above via `KNOWN_HEIGHT_CM` in section 7-11 (same worker call, run twice: once
with `known_height_cm=None` for the raw result, once with it set for the calibrated result — both
are preserved separately, never overwritten). The scale factor is a single uniform multiplier
(`known_height_cm / raw_body_height_cm`) applied to every mesh vertex; it corrects overall scale
only and does **not** alter body proportions to match any customer-reported measurement other
than height itself. See `rescale.py`'s docstring for the full list of what this does not correct
(camera perspective, posture, non-uniform proportion errors).

## 13. Save results

In [ ]:
import datetime, pathlib

RESULTS_DIR_RUNTIME = '/content/results'  # ephemeral Colab runtime only
os.makedirs(RESULTS_DIR_RUNTIME, exist_ok=True)

output_record = {
    'subject_id': pathlib.Path(input_image_path).stem,
    'used_public_sample_image': used_sample_image,
    'checkpoint_used': primary['checkpoint_repo'] if primary['checkpoint_downloaded'] else None,
    'checkpoint_size_bytes': primary['checkpoint_size_bytes'],
    'sam3d_inference_status': 'ok' if primary['sam3d_inference_ok'] else 'failed',
    'person_detected': primary['person_detected'],
    'sam3d_output_schema': primary['output_schema'],
    'raw_body_height_cm': raw_body_height_cm,
    'raw_measurements_cm': raw_measurements_cm,
    'known_height_calibration_applied': calibrated_measurements_cm is not None,
    'known_height_cm': KNOWN_HEIGHT_CM,
    'calibrated_scale_factor': calibrated_scale_factor,
    'calibrated_measurements_cm': calibrated_measurements_cm,
    'warnings': primary['warnings'],
    'failures': primary['failures'],
    'generated_at_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'note': (
        'Technical smoke test only -- NOT an anthropometric accuracy claim. A successful '
        'measurement here proves clad-body computed a circumference/length from the '
        'SAM-3D-Body-predicted mesh, not that the mesh matches the true photographed body.'
    ),
}

runtime_output_path = f'{RESULTS_DIR_RUNTIME}/measurement_output.json'
with open(runtime_output_path, 'w') as f:
    json.dump(output_record, f, indent=2)
print('Saved (runtime-only, not committed to git):', runtime_output_path)

if used_sample_image:
    repo_results_dir = f'{EXPERIMENT_DIR}/results'
    os.makedirs(repo_results_dir, exist_ok=True)
    with open(f'{repo_results_dir}/measurement_output.json', 'w') as f:
        json.dump(output_record, f, indent=2)
    print('Also written into the cloned repo (public sample image only -- safe to commit):',
          f'{repo_results_dir}/measurement_output.json')
    print('This file is written to your LOCAL clone in this Colab runtime. It is not committed ')
    print('or pushed automatically -- git add/commit/push yourself if you want to keep it.')
else:
    print('Personal image used -- result kept in the Colab runtime only, NOT written into the cloned repo.')

## 14. Compute/runtime summary (measured, not estimated)

In [ ]:
compute_summary = {
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'peak_vram_mb': primary['peak_vram_mb'],
    'system_ram_gb': round(ram_gb, 1) if ram_gb else None,
    'clad_env_setup_time_s': clad_env_setup_time_s,
    'checkpoint_size_bytes': primary['checkpoint_size_bytes'],
    'checkpoint_download_time_s': primary['checkpoint_download_time_s'],
    'sam3d_load_time_s': primary['sam3d_load_time_s'],
    'sam3d_inference_time_s': primary['sam3d_inference_time_s'],
    'measurement_extraction_time_s': primary['measurement_extraction_time_s'],
}
print(json.dumps(compute_summary, indent=2))

## 15. Optional: second-checkpoint comparison (only if cheap — off by default)

Internal consistency check only (same image, two checkpoints) — **not** an accuracy comparison;
there is no ground truth here. Set `RUN_SECOND_CHECKPOINT_COMPARISON = True` to enable. Does not
affect the `state`/decision-gate below either way — the gate reflects the primary checkpoint only.

In [ ]:
RUN_SECOND_CHECKPOINT_COMPARISON = False
SECOND_CHECKPOINT_REPO = 'facebook/sam-3d-body-vith'

secondary = None
if RUN_SECOND_CHECKPOINT_COMPARISON and access_status.get(SECOND_CHECKPOINT_REPO) == 'accessible':
    secondary = run_full_pipeline_for_checkpoint(SECOND_CHECKPOINT_REPO, input_image_path, record_to_state=False)
    sec_meas = secondary['raw_result']['measurements_cm'] if secondary['raw_result'] else None
    compare_keys = ['height_cm', 'bust_cm', 'waist_cm', 'hip_cm', 'shoulder_width_cm']
    print(f"{'key':16s} {PRIMARY_CHECKPOINT_REPO:28s} {SECOND_CHECKPOINT_REPO:24s}")
    for k in compare_keys:
        v1 = raw_measurements_cm.get(k) if raw_measurements_cm else None
        v2 = sec_meas.get(k) if sec_meas else None
        print(f'{k:16s} {str(v1):28s} {str(v2):24s}')
    print('runtime (s):', primary['sam3d_inference_time_s'], 'vs', secondary['sam3d_inference_time_s'])
    print('peak VRAM (MB):', primary['peak_vram_mb'], 'vs', secondary['peak_vram_mb'])
else:
    print('Second-checkpoint comparison skipped (disabled by default, or access to',
          SECOND_CHECKPOINT_REPO, 'was not confirmed in section 5). This is expected --',
          'per Task 03 section 12, only run this if cheap; deferred otherwise.')

## 16. Decision gate

In [ ]:
gate, reason = classify(state)
print('DECISION GATE:', gate.value, '-', gate.name)
print('Reason:', reason)
print()
print('Recorded failure categories:', state.failure_categories or 'none')
print()
print('State snapshot:')
for field_name in ['gpu_available', 'hf_auth_ok', 'checkpoint_downloaded', 'dependencies_installed',
                    'sam3d_inference_ok', 'mhr_schema_valid', 'mhr_reconstruction_ok', 'clad_body_measure_ok']:
    print(f'  {field_name}: {getattr(state, field_name)}')
print(f'  measurements produced: {bool(state.measurements)}')